<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_5_GroupBy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.5 — GroupBy and Aggregation

## Learning Objectives

By the end of this notebook you should be able to:

- Explain the **split–apply–combine** pattern in plain language
- Use `.groupby()` to compute group-level summaries
- Apply aggregation functions: `.mean()`, `.sum()`, `.count()`, `.min()`, `.max()`
- Use `.agg()` to compute several summaries at once with named output columns
- Group by two keys simultaneously and interpret the result
- Sort and clean up grouped results with `.reset_index()`

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

## The Question This Notebook Answers

Every analysis so far has worked on the full 887 passengers, or on a filtered subset like "first-class women." But the most revealing questions in the Titanic data are **comparisons between groups**:

- Did women survive at a higher rate than men?
- Were first-class passengers more likely to survive than third-class?
- Did passengers who traveled with family have a different survival rate than those who traveled alone?

To answer these, you need to compute a summary (like a survival rate) *separately for each group*, then compare. That's what **GroupBy** does.

## 1. The Split–Apply–Combine Pattern

GroupBy works in three steps that always happen in the same order:

**Split** — divide the DataFrame into groups based on the values in one column. If you group by `sex`, you get two groups: all male rows and all female rows.

**Apply** — compute something within each group. For example, compute the mean of the `survived` column within each group.

**Combine** — assemble the per-group results into a single output Series or DataFrame.

```
Full dataset (887 rows)
        sex   survived   fare
        male       0     7.25
        female     1    71.28
        female     1     7.92
        male       0    53.10
        ...

      ▼  SPLIT by "sex"

  GROUP: male         GROUP: female
  survived  fare      survived  fare
      0     7.25          1    71.28
      0    53.10          1     7.92
      ...                ...

      ▼  APPLY: mean("survived")

  male → 0.19          female → 0.74

      ▼  COMBINE

  sex       survived
  female    0.74
  male      0.19
```

## 2. Your First GroupBy

The syntax is: `df.groupby("column")["column_to_aggregate"].aggregation_function()`

Let's answer the first question: did women survive at a higher rate than men?

In [ ]:
# .mean() on a 0/1 column gives the proportion that equals 1 — the survival rate
df.groupby("sex")["survived"].mean()

Women survived at 74%, men at 19%. That's a dramatic difference, and it's not a coincidence — "women and children first" was the evacuation protocol.

Now the second question: did survival differ by class?

In [ ]:
df.groupby("pclass")["survived"].mean().round(2)

First-class passengers survived at 63%; third-class at only 24%. Part of this is proximity — third-class cabins were below decks, and by the time passengers made it to the top the lifeboats were gone. Part of it may also reflect who was directed toward the lifeboats first.

Let's also check average fare by class — does the fare data corroborate the class labels?

In [ ]:
df.groupby("pclass")["fare"].mean().round(2)

Yes — first-class passengers paid on average £87, third-class passengers £13. The class variable is real.

## 3. When You Need More Than One Number Per Group

A survival rate is useful, but it's just one number. To understand a group properly you often want several summaries at once — how many passengers were in each group? What was the youngest? The oldest? The average fare?

`.agg()` computes multiple aggregations in a single call.

In [ ]:
# How many passengers per class, and what was their average fare?
df.groupby("pclass")["fare"].agg(["count", "mean", "median", "min", "max"]).round(2)

In [ ]:
# Named aggregations — give each output column a meaningful name
df.groupby("pclass").agg(
    passengers    = ("survived", "count"),
    survival_rate = ("survived", "mean"),
    avg_fare      = ("fare",     "mean"),
    avg_age       = ("age",      "mean"),
).round(2)

Reading this table: first-class passengers were older on average (39 years vs 25 for third class), paid more, and survived at a much higher rate. These three facts together suggest that class, age, and survival are all correlated — something we'll explore further in the next notebook.

## 4. Grouping by Two Keys

The next natural question: is the gender survival gap consistent across all three classes, or is it stronger in some classes than others?

To answer that, you need to group by *both* class and sex simultaneously.

In [ ]:
df.groupby(["pclass", "sex"])["survived"].mean().round(2)

Every sex/class combination shows the same pattern: women survived at much higher rates than men. But notice that even third-class women (49%) survived at more than double the rate of first-class men (37%). Being female was the dominant factor — more important than class.

In [ ]:
# Full picture: count and survival rate by class and sex
df.groupby(["pclass", "sex"]).agg(
    count         = ("survived", "count"),
    survival_rate = ("survived", "mean"),
).round(2)

## 5. Cleaning Up the Result

After a groupby, the grouping columns (`pclass`, `sex`) become the **index** of the result — not regular columns. This can make the output awkward to work with. `.reset_index()` promotes them back to regular columns.

In [ ]:
# Without reset_index: pclass is the index
result = df.groupby("pclass")["survived"].mean()
print("Type:", type(result))
print("Index:", result.index.tolist())
print(result)

In [ ]:
# With reset_index: pclass is a regular column you can sort, rename, filter
result_df = (
    df.groupby("pclass")["survived"].mean()
    .reset_index()
    .rename(columns={"survived": "survival_rate"})
    .sort_values("survival_rate", ascending=False)
)
result_df

## 6. Putting It Together

Let's answer the compound question: *For each combination of class and sex, how many passengers were there, what fraction survived, how old were they on average, and what did they pay?*

In [ ]:
summary = (
    df.groupby(["pclass", "sex"])
    .agg(
        passengers    = ("survived", "count"),
        survival_rate = ("survived", "mean"),
        avg_age       = ("age",      "mean"),
        avg_fare      = ("fare",     "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(["pclass", "sex"])
)
summary

In [ ]:
# Highlight the gender gap in a readable format
print("Survival rates by class and sex\n")
for cls in sorted(df["pclass"].unique()):
    male   = df.loc[(df["pclass"]==cls) & (df["sex"]=="male"),   "survived"].mean()
    female = df.loc[(df["pclass"]==cls) & (df["sex"]=="female"), "survived"].mean()
    print(f"Class {cls}:  men {male:.0%}  |  women {female:.0%}")

## Your Turn

1. What is the average age of passengers who survived vs those who did not? Does age seem to be a factor?
2. Did passengers traveling with family (`sibsp > 0` or `parch > 0`) have a higher survival rate than those traveling alone? Compute the rate for each group.
3. Build a full summary table for each value of `sex`: count, average age, average fare, survival rate. Sort by survival rate.
4. Is there a pattern between the number of siblings/spouses (`sibsp`) and survival rate? Compute survival rate grouped by `sibsp` value and describe what you see.

In [ ]:
# Your code here